In [1]:
# ============================================================
# Notebook : 02_pcmci.ipynb
# Purpose  : Step 3 (Causal Stability) - part 1: temporal causal
#            discovery. Run PCMCI+ (ParCorr, single lag) per household
#            on the twelve continuous sensor channels, separate the
#            trivial autoregressive (self-lag) structure from the
#            cross-variable precursor links, aggregate the cross links
#            across households, and define a per-category causal-path
#            variable set that later constrains the counterfactuals of
#            Step 4. The category Skin-temp (ST) has NO dedicated sensor
#            channel, so its precursors can only be read from the lag-1
#            association with the label indicator - a weaker, non-causal
#            basis that Step 4/5 exploits to expose intervention bypass.
# Note     : relation strength itself is not a pass/fail criterion; the
#            aim is to narrow candidate intervention variables, and only
#            the sign and rank of links are intended to be reproducible.
# Design   : Single code cell. No absolute path is printed.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "axes.unicode_minus": False,
    "savefig.dpi": 600, "figure.dpi": 120, "axes.edgecolor": "0.15", "grid.color": "0.85",
})


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
V = meta["sensor_cont"]
DEFINING = meta["category_defining_sensor"]
ABN = meta["abnormal_categories"]

# ---- PCMCI+ configuration ----
TAU_MAX = 1
PC_ALPHA = 0.01
MIN_ROWS = 200        # households with fewer records are not used for discovery
RECUR = 0.25          # a cross link is "recurring" if present in >= 25% of households

# ============================================================
# Run PCMCI+ per household and collect lag-1 directed links.
# ============================================================
households = sorted(df["HH_ID"].unique())
used_hh = []
link_val = {}   # (src, tgt) -> list of partial-corr values

for hh in households:
    g = df[df["HH_ID"] == hh].sort_values("Timestamp")
    if len(g) < MIN_ROWS:
        continue
    X = g[V].astype(float).values
    sd = X.std(axis=0)
    keep = sd > 1e-8
    names = [v for v, k in zip(V, keep) if k]
    Xs = (X[:, keep] - X[:, keep].mean(0)) / X[:, keep].std(0)
    dataframe = pp.DataFrame(Xs, var_names=names)
    pcmci = PCMCI(dataframe=dataframe, cond_ind_test=ParCorr(), verbosity=0)
    res = pcmci.run_pcmciplus(tau_min=0, tau_max=TAU_MAX, pc_alpha=PC_ALPHA)
    graph, vm = res["graph"], res["val_matrix"]
    used_hh.append(hh)
    n = len(names)
    for i in range(n):
        for j in range(n):
            if graph[i, j, 1] == "-->":        # X_i(t-1) --> X_j(t)
                link_val.setdefault((names[i], names[j]), []).append(vm[i, j, 1])

n_used = len(used_hh)


def agg_rows(pairs):
    out = []
    for key, vals in pairs.items():
        out.append({
            "source": key[0], "target": key[1],
            "freq_hh": len(vals), "freq_share": round(len(vals) / n_used, 3),
            "mean_parcorr": round(float(np.mean(vals)), 4),
            "median_parcorr": round(float(np.median(vals)), 4),
            "mean_abs": round(float(np.mean(np.abs(vals))), 4),
            "sign_consistency": round(float(np.mean(np.sign(vals) == np.sign(np.mean(vals)))), 3),
        })
    return pd.DataFrame(out)

# ---- Separate autoregressive (self) from cross-variable links ----
self_pairs = {k: v for k, v in link_val.items() if k[0] == k[1]}
cross_pairs = {k: v for k, v in link_val.items() if k[0] != k[1]}

self_links = agg_rows(self_pairs).sort_values("mean_abs", ascending=False).reset_index(drop=True)
cross = agg_rows(cross_pairs).sort_values(["freq_hh", "mean_abs"], ascending=False).reset_index(drop=True)
cross.to_csv(TAB / "table_all_cross_links.csv", index=False, encoding="utf-8-sig")

recurring = cross[cross["freq_share"] >= RECUR].reset_index(drop=True)
recurring.to_csv(TAB / "table_stable_cross_links.csv", index=False, encoding="utf-8-sig")

max_cross_abs = cross["mean_abs"].max()
max_self_abs = self_links["mean_abs"].max()

# ============================================================
# Define per-category causal-path variable set (for Steps 4-5).
#   HR / BR / SP : defining sensor + its recurring cross-parents.
#   ST           : lag-1 label-associated sensors (no defining channel).
#   ER           : raw signal, excluded from counterfactual modelling.
# ============================================================
causal_paths, prec_rows = {}, []
for c in ABN:
    sensor = DEFINING[c]
    if sensor in V:                                  # HR, BR, SP
        pars = recurring[recurring["target"] == sensor].sort_values("mean_abs", ascending=False)
        parents = list(pars["source"])
        path = [sensor] + parents
        detail = f"{sensor}[self]" + ("".join(
            f", {r.source}({r.mean_parcorr:+.3f})" for r in pars.itertuples()) or " (no recurring cross-parent)")
        basis = "PCMCI lag-1: defining sensor + recurring cross-parents"
    elif c == "ST":                                  # no defining channel
        assoc = {}
        for hh in used_hh:
            g = df[df["HH_ID"] == hh].sort_values("Timestamp")
            y = g["cat_ST"].values[1:]
            for v in V:
                x = g[v].astype(float).values[:-1]
                if x.std() > 1e-8 and y.std() > 1e-8:
                    assoc.setdefault(v, []).append(np.corrcoef(x, y)[0, 1])
        amean = {v: float(np.nanmean(a)) for v, a in assoc.items()}
        top = sorted(amean.items(), key=lambda kv: -abs(kv[1]))[:3]
        path = [v for v, _ in top]
        detail = ", ".join(f"{v}({r:+.3f})" for v, r in top)
        basis = "label-association only (NO defining sensor channel)"
        sensor = "(none)"
    else:                                            # ER
        path = []
        detail = "(direct raw signal; not modelled)"
        basis = "raw emergency signal"
        sensor = "Button/Shout"
    causal_paths[c] = path
    prec_rows.append({"category": c, "defining_sensor": sensor, "basis": basis,
                      "n_path_vars": len(path), "causal_path": detail})

precursors_tbl = pd.DataFrame(prec_rows)
precursors_tbl.to_csv(TAB / "table_category_precursors.csv", index=False, encoding="utf-8-sig")
pd.to_pickle({"causal_paths": causal_paths, "used_hh": used_hh,
              "max_cross_abs": max_cross_abs}, DATA / "causal_paths.pkl")

# ============================================================
# Figure 5. Lag-1 cross-link frequency heatmap (source -> target).
# ============================================================
F = pd.DataFrame(0.0, index=V, columns=V)
for r in cross.itertuples():
    F.loc[r.source, r.target] = r.freq_share
fig, ax = plt.subplots(figsize=(6.6, 5.6))
sns.heatmap(F, cmap="Greys", vmin=0, vmax=1, linewidths=0.4, linecolor="0.7",
            cbar_kws={"label": "Share of households with lag-1 cross-link"}, ax=ax)
ax.set_xlabel("Target  (t)")
ax.set_ylabel("Source  (t-1)")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
save_fig(fig, "fig05_cross_link_frequency")

# ============================================================
# Figure 6. Aggregate causal DAG of recurring cross-links (grayscale).
#           Edge darkness and width encode across-household frequency;
#           self-lags are omitted (all channels are strongly autoregressive).
# ============================================================
G = nx.DiGraph()
G.add_nodes_from(V)
for r in recurring.itertuples():
    G.add_edge(r.source, r.target, w=r.freq_share)
fig, ax = plt.subplots(figsize=(7.2, 6.0))
pos = nx.spring_layout(G, seed=7, k=1.2)
nx.draw_networkx_nodes(G, pos, node_color="0.9", edgecolors="black", node_size=1500, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
if G.number_of_edges() > 0:
    ws = np.array([G[u][v]["w"] for u, v in G.edges()])
    nx.draw_networkx_edges(G, pos, width=0.5 + 3 * ws,
                           edge_color=[str(max(0.0, 0.75 - w)) for w in ws],
                           arrowsize=12, connectionstyle="arc3,rad=0.08", ax=ax)
ax.axis("off")
save_fig(fig, "fig06_aggregate_dag")

# ---- Console summary ----
print("=== Step 3 (part 1): PCMCI+ causal discovery ===")
print(f"Households used for discovery : {n_used} / {df['HH_ID'].nunique()} (>= {MIN_ROWS} records)")
print(f"Autoregressive self-lags      : all {len(self_links)} channels; "
      f"max |partial corr| = {max_self_abs:.3f} (trivially strong, not a precursor)")
print(f"Distinct cross-variable links : {len(cross)}")
print(f"Recurring cross-links (>= {int(RECUR*100)}%) : {len(recurring)}")
print(f"Max cross-link |partial corr| : {max_cross_abs:.4f}  "
      f"(conventionally weak; only sign/rank are intended to reproduce)")
print()
print("Recurring cross-variable lag-1 links:")
print(recurring.to_string(index=False))
print()
print("Per-category causal-path variable set:")
print(precursors_tbl.to_string(index=False))
print()
print("Figures saved: fig05_cross_link_frequency, fig06_aggregate_dag  (png + pdf, 600 dpi)")
print("Tables saved : table_all_cross_links, table_stable_cross_links, table_category_precursors")
print("Artifact     : causal_paths.pkl  (per-category on-path variables for Steps 4-5)")


=== Step 3 (part 1): PCMCI+ causal discovery ===
Households used for discovery : 23 / 25 (>= 200 records)
Autoregressive self-lags      : all 12 channels; max |partial corr| = 0.814 (trivially strong, not a precursor)
Distinct cross-variable links : 42
Recurring cross-links (>= 25%) : 9
Max cross-link |partial corr| : 0.1184  (conventionally weak; only sign/rank are intended to reproduce)

Recurring cross-variable lag-1 links:
     source      target  freq_hh  freq_share  mean_parcorr  median_parcorr  mean_abs  sign_consistency
 SleepScore  SleepPhase       16       0.696       -0.0708         -0.0624    0.0708               1.0
Illuminance Temperature       11       0.478        0.0792          0.0886    0.0792               1.0
        CO2        TVOC       10       0.435        0.0109          0.0109    0.0566               0.5
Activity_IR  SleepPhase        9       0.391        0.0580          0.0496    0.0580               1.0
        CO2    Humidity        7       0.304        0.